# The Modifiable Areal Unit Problem

**DS4DH Practice Pack · Module 09 — Geospatial Analysis**

*Technique:* How the choice of spatial unit and weighting changes the answer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/09b_maup.ipynb)

Data: `merged_dataset.csv`, `city_summary.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# This notebook reads the CSVs sitting next to it. In Colab you will be asked
# to upload them from the pack's data/ folder.
NEEDED = ['merged_dataset.csv', 'city_summary.csv']

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('merged_dataset.csv')
df_city  = pd.read_csv('city_summary.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

The Modifiable Areal Unit Problem is the observation that statistics computed
over spatial units depend on how those units were drawn — and the boundaries were
drawn for administrative reasons that have nothing to do with your question.

It has two parts:

- **Scale** — aggregating to CSDs vs CMAs gives different numbers.
- **Zoning** — at the same scale, drawing the boundaries differently gives
  different numbers again.

This is not a flaw to be corrected. It is a property of all areal data, and the
response is to report which units you used and check whether your conclusion
survives changing them.

In [ ]:
# One row per Census Subdivision.
#   • rows with no csd_code are CMA-level and Canada-level aggregates, not CSDs
#   • each CSD appears 3x (Immigrant / Non-immigrants / Total Immigrant Status)
# Keeping either would silently double- or triple-count places.
csd = df.dropna(subset=['csd_code'])
base = csd[(csd['immigrant_status'] == 'Total Immigrant Status')
           & (csd['cma'].isin(CITIES))].copy()

print(f'{len(df):>4} rows in the raw file')
print(f'{len(csd):>4} after dropping CMA/Canada aggregate rows')
print(f'{len(base):>4} CSDs in the four cities (one row each)')

In [ ]:
# Two ways to get "the" average housing burden for a city.
d = base.dropna(subset=['Total', 'tot_pop'])

print(f'{"City":<12}{"n CSDs":>8}{"equal-weighted":>17}{"pop-weighted":>15}{"diff":>9}')
print('-' * 61)
for city in CITIES:
    g = d[d['cma'] == city]
    equal = g['Total'].mean()
    weighted = np.average(g['Total'], weights=g['tot_pop'])
    print(f'{city:<12}{len(g):>8}{equal:>17.2f}{weighted:>15.2f}{equal - weighted:>+9.2f}')

Both columns are correct. They answer different questions:

- **Equal-weighted** — "what is the average municipality like?" Every place counts
  once, so a village of 600 has the same say as a city of 2.6 million.
- **Population-weighted** — "what is the average *person's* experience?" Every
  household counts once.

For a policy brief about people, the weighted figure is almost always the right
one. For a brief about municipal governments, the unweighted one may be.

In [ ]:
# The third answer: the CMA-level published row, which is neither of the above.
cma_rows = df[df['csd_code'].isna() & (df['immigrant_status'] == 'Total Immigrant Status')]

print(f'{"City":<12}{"published CMA":>15}{"pop-weighted":>15}{"equal-weighted":>17}')
print('-' * 60)
for city in CITIES:
    pub = cma_rows[cma_rows['cma'] == city]['Total']
    g = d[d['cma'] == city]
    p = float(pub.iloc[0]) if len(pub) else float('nan')
    print(f'{city:<12}{p:>15.2f}'
          f'{np.average(g["Total"], weights=g["tot_pop"]):>15.2f}'
          f'{g["Total"].mean():>17.2f}')
print()
print('Three defensible numbers for the same city. Any of them can be quoted')
print('honestly; quoting one without saying which is what misleads.')

### 🔧 Your turn 1

The published CMA figure and the population-weighted CSD figure should be close,
but are not identical.

Name two reasons why aggregating from the parts would not exactly reproduce the
published whole. (Hint: which CSDs are missing from your sum, and what is the
denominator of a ratio of ratios?)

## Scale — the same data at two resolutions

Aggregating hides variation. This measures how much.

In [ ]:
print(f'{"City":<12}{"CMA-level value":>17}{"CSD min":>10}{"CSD max":>10}{"range":>9}')
print('-' * 58)
for city in CITIES:
    g = d[d['cma'] == city]
    w = np.average(g['Total'], weights=g['tot_pop'])
    print(f'{city:<12}{w:>17.2f}{g["Total"].min():>10.1f}'
          f'{g["Total"].max():>10.1f}{g["Total"].max() - g["Total"].min():>9.1f}')
print()
print('A single CMA number stands in for a range of 15-25 percentage points.')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
for i, city in enumerate(CITIES):
    g = d[d['cma'] == city]
    ax.scatter(np.full(len(g), i) + np.random.default_rng(1).normal(0, 0.06, len(g)),
               g['Total'], s=20, alpha=0.5)
    w = np.average(g['Total'], weights=g['tot_pop'])
    ax.plot([i - 0.25, i + 0.25], [w, w], color='#E8663D', lw=3)
ax.set_xticks(range(len(CITIES)))
ax.set_xticklabels(CITIES)
ax.set_ylabel('Total STIR (%)')
ax.set_title('Each dot is a CSD; the bar is the population-weighted CMA figure')
plt.tight_layout()
plt.show()

## Zoning — regrouping at the same scale

You cannot redraw census boundaries, but you can regroup the same CSDs by a
different rule and see how much the answer moves. Grouping by population band
instead of by city is a legitimate alternative partition of exactly the same data.

In [ ]:
d2 = d.copy()
d2['size_band'] = pd.cut(d2['tot_pop'],
                         bins=[0, 5000, 25000, 100000, np.inf],
                         labels=['<5k', '5-25k', '25-100k', '100k+'])

by_city = d2.groupby('cma', observed=True)['Total'].mean()
by_size = d2.groupby('size_band', observed=True)['Total'].mean()

print('Same 155 CSDs, two different partitions:')
print()
print('grouped by city:')
print(by_city.round(2).to_string())
print()
print('grouped by population band:')
print(by_size.round(2).to_string())
print()
print(f'spread across cities:      {by_city.max() - by_city.min():.2f} pp')
print(f'spread across size bands:  {by_size.max() - by_size.min():.2f} pp')

### 🔧 Your turn 2

Change the `bins` to `[0, 10000, 50000, np.inf]` (three bands instead of four).

Does the ordering of the bands stay the same? Does the spread? You have changed
nothing about the world — only how you cut it.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** Two reasons. First, the CSDs with suppressed data are missing
from your sum but were included in the published CMA total, so you are
aggregating a subset. Second, STIR is a *ratio*, and the average of ratios is not
the ratio of averages — properly reconstructing the CMA figure means summing
shelter costs and summing incomes, then dividing, rather than averaging the
per-CSD ratios. Population weighting gets you closer but not exactly there.

**Your turn 2.** The ordering across size bands is usually stable — larger, more
urban municipalities carry higher burden — but the spread changes noticeably with
the cut points, and with three bands the extremes are diluted. That is MAUP in one
cell: the same 155 observations, unchanged, produce a different headline number
depending on a binning decision you made in a moment and could easily not have
documented.

</details>

## Where this stops

You know your city averages are one defensible choice among several. What you
have not measured is how *unequal* things are inside each city — which is often
the more policy-relevant question. That is the next notebook.